In [89]:
import pandas as pd
import numpy as np

In [90]:
df=pd.read_csv("diabetes_dataset.csv")

df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,2.0,3.0,4.0,2.0,888.0,888.0,3.0
1,1.0,3.0,4.0,88.0,888.0,888.0,3.0
2,2.0,3.0,2.0,2.0,101.0,101.0,3.0
3,2.0,4.0,4.0,88.0,888.0,230.0,3.0
4,1.0,4.0,4.0,25.0,301.0,888.0,3.0


**Split Training and Test Set**

In [91]:
#seperate training and test data
X, y = df.loc[:, ~df.columns.isin(['DIABETE4'])], df.loc[:, df.columns.isin(['DIABETE4'])]

In [92]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=1)

In [93]:
print('training set size: {}'.format(len(X_train)))
print('test set size: {}'.format(len(X_test)))

training set size: 80416
test set size: 20104


**SVM Model Specific Pre-Processing**

In [94]:
#convert labels to -1 (og was 3) and +1 (og was 1)
y_train_svm = np.where(y_train.to_numpy().ravel() == 1, 1, -1)
y_test_svm = np.where(y_test.to_numpy().ravel() == 1, 1, -1)

In [95]:
#preserve raw train/test splits
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()
y_train_raw = y_train.copy()
y_test_raw = y_test.copy()

#normalize using training stats
mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
std[std == 0] = 1

X_train_scaled = (X_train_raw - mean) / std
X_test_scaled = (X_test_raw - mean) / std

**Building a SVM Model**

In [96]:
#define the model

class SVM(object):
    def __init__(self, learning_rate=0.001, num_iterations=1000, lambda_param=0.01, positive_class_weight=1.0):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.num_iterations = num_iterations
        self.W = None
        self.b = 0
        self.likelihood_history = []
        self.positive_class_weight = positive_class_weight

    def predict_score(self, X):
        return np.dot(X, self.W) + self.b

    def compute_hinge_loss(self, X, Y):
        margins = 1 - Y * self.predict_score(X)
        margins = np.maximum(0, margins)
        loss = np.mean(margins) + (self.lambda_param / 2) * np.sum(self.W ** 2)
        return loss

    def update_weights(self):
        num_examples = self.X.shape[0]

        scores = self.predict_score(self.X)
        indicator = (self.Y * scores) < 1
        indicator = indicator.astype(float)

        #give more weight to the positive class - the diabetes class
        class_weights = np.where(self.Y == 1, self.positive_class_weight, 1.0)

        weighted_indicator = indicator * class_weights

        dW = (-np.dot(self.X.T, (self.Y * weighted_indicator)) + 2 * self.lambda_param * self.W) / num_examples
        db = -np.sum(self.Y * weighted_indicator) / num_examples

        self.W -= self.learning_rate * dW
        self.b -= self.learning_rate * db

        loss = self.compute_hinge_loss(self.X, self.Y)
        self.likelihood_history.append(loss)

    def fit(self, X, Y):
        self.X = np.asarray(X, dtype=float)
        self.Y = np.asarray(Y, dtype=float).reshape(-1)

        num_features = self.X.shape[1]
        self.W = np.zeros(num_features)
        self.b = 0
        self.likelihood_history = []

        for _ in range(self.num_iterations):
            self.update_weights()

    def predict(self, X):
        scores = self.predict_score(X)
        return np.where(scores >= 0, 1, -1)

In [97]:
#train
svm_model = SVM(learning_rate=0.001, num_iterations=1000, lambda_param=0.01)
svm_model.fit(X_train_scaled, y_train_svm)

In [98]:
#predict
y_train_pred = svm_model.predict(X_train_scaled)
y_test_pred = svm_model.predict(X_test_scaled)

**Classification Accuracy, Precision, Recall, and F1 Score**

In [99]:
#accuracy 
train_accuracy = np.mean(np.asarray(y_train_pred).ravel() == np.asarray(y_train_svm).ravel())
test_accuracy = np.mean(np.asarray(y_test_pred).ravel() == np.asarray(y_test_svm).ravel())

print("train accuracy:", train_accuracy)
print("test accuracy:", test_accuracy)

train accuracy: 0.8551905093513729
test accuracy: 0.857391563867887


In [100]:
def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    tn = np.sum((y_true == -1) & (y_pred == -1))

    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
    accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) != 0 else 0

    return accuracy, precision, recall, f1_score, tp, fp, fn, tn


In [101]:
accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test_svm, y_test_pred)

print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("f1 score:", f1_score)
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)


accuracy: 0.857391563867887
precision: 0
recall: 0.0
f1 score: 0
TP: 0
FP: 0
FN: 2867
TN: 17237


In [102]:
print("predicted class distribution:", np.unique(y_test_pred, return_counts=True))

predicted class distribution: (array([-1]), array([20104]))


**SVM Imbalance Experiments**

In [103]:
feature_cols = X_train_raw.columns.tolist()
target_col = "DIABETE4"

undersampling_df = pd.read_csv("data_undersampling.csv")
oversampling_df = pd.read_csv("data_oversampling.csv")
smote_df = pd.read_csv("data_smote.csv")

X_train_original = X_train_raw.copy()
y_train_original = y_train_raw.copy()

X_train_under = undersampling_df[feature_cols].copy()
y_train_under = undersampling_df[[target_col]].copy()

X_train_over = oversampling_df[feature_cols].copy()
y_train_over = oversampling_df[[target_col]].copy()

X_train_smote = smote_df[feature_cols].copy()
y_train_smote = smote_df[[target_col]].copy()

X_train_original_scaled = (X_train_original - mean) / std
X_train_under_scaled = (X_train_under - mean) / std
X_train_over_scaled = (X_train_over - mean) / std
X_train_smote_scaled = (X_train_smote - mean) / std

y_train_original_svm = np.where(y_train_original.to_numpy().ravel() == 1, 1, -1)
y_train_under_svm = np.where(y_train_under.to_numpy().ravel() == 1, 1, -1)
y_train_over_svm = np.where(y_train_over.to_numpy().ravel() == 1, 1, -1)
y_train_smote_svm = np.where(y_train_smote.to_numpy().ravel() == 1, 1, -1)
y_test_svm = np.where(y_test.to_numpy().ravel() == 1, 1, -1)

datasets = {
    "original": (X_train_original_scaled, y_train_original_svm),
    "undersampling": (X_train_under_scaled, y_train_under_svm),
    "oversampling": (X_train_over_scaled, y_train_over_svm),
    "smote": (X_train_smote_scaled, y_train_smote_svm),
}

weights_to_try = [1.0, 2.0, 3.0]

results = []

for dataset_name, (X_train_dataset, y_train_dataset) in datasets.items():
    for weight in weights_to_try:
        svm_model = SVM(
            learning_rate=0.001,
            num_iterations=1000,
            lambda_param=0.01,
            positive_class_weight=weight,
        )
        svm_model.fit(X_train_dataset, y_train_dataset)
        y_test_pred = svm_model.predict(X_test_scaled)

        accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test_svm, y_test_pred)

        results.append({
            "dataset": dataset_name,
            "weight": weight,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
        })

results_df = pd.DataFrame(results, columns=["dataset", "weight", "accuracy", "precision", "recall", "f1_score", "tp", "fp", "fn", "tn"])
results_df = results_df.sort_values(["dataset", "weight"]).reset_index(drop=True)
results_df


,dataset,weight,accuracy,precision,recall,f1_score,tp,fp,fn,tn
0,original,1.0,0.857392,0.000000,0.000000,0.000000,0,0,2867,17237
1,original,2.0,0.848189,0.399128,0.127660,0.193446,366,551,2501,16686
2,original,3.0,0.786709,0.308232,0.398326,0.347535,1142,2563,1725,14674
3,oversampling,1.0,0.610127,0.230861,0.743634,0.352338,2132,7103,735,10134
4,oversampling,2.0,0.388331,0.179121,0.918033,0.299755,2632,12062,235,5175
5,oversampling,3.0,0.223239,0.153353,0.983607,0.265337,2820,15569,47,1668
6,smote,1.0,0.610724,0.230986,0.742588,0.352367,2129,7088,738,10149
7,smote,2.0,0.391962,0.179884,0.916986,0.300767,2629,11986,238,5251
8,smote,3.0,0.223289,0.153474,0.984653,0.265557,2823,15571,44,1666
9,undersampling,1.0,0.606944,0.229853,0.747122,0.351551,2142,7177,725,10060


**Final Model Selection and Development Prep**

Oversampling with weight = 1.0 gave the best balance of precision and recall and had the highest F1 score. Increasing the weight boosted recall but also created too many false positives. Since all three methods performed similarly, I chose oversampling because it’s simpler and keeps all the data.

In [104]:
svm_final = SVM(
    learning_rate=0.001,
    num_iterations=1000,
    lambda_param=0.01,
    positive_class_weight=1.0
)
svm_final.fit(X_train_over_scaled, y_train_over_svm)


In [105]:
# Use existing held-out scaled test set if final aliases are not yet defined
X_test_final = X_test_scaled
y_test_final_svm = y_test_svm

y_test_pred_final = svm_final.predict(X_test_final)
accuracy, precision, recall, f1_score, tp, fp, fn, tn = classification_metrics(y_test_final_svm, y_test_pred_final)

print("final model evaluation (oversampling + weight 1.0)")
print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("f1 score:", f1_score)
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)

final model evaluation (oversampling + weight 1.0)
accuracy: 0.6101273378432153
precision: 0.2308608554412561
recall: 0.7436344611091733
f1 score: 0.3523384564534788
TP: 2132
FP: 7103
FN: 735
TN: 10134


In [106]:
y_test_pred_original = np.where(y_test_pred_final == 1, 1, 3)
y_test_pred_original[:10]

array([1, 1, 1, 3, 3, 3, 1, 1, 1, 3])

In [107]:
model_params = {
    "W": svm_final.W,
    "b": svm_final.b,
    "mean": mean,
    "std": std
}

In [108]:
def predict_new_data(X_new, model_params):
    X_new = np.asarray(X_new, dtype=float)
    #std is assumed to have been cleaned earlier (zero values handled)
    X_new_scaled = (X_new - model_params["mean"]) / model_params["std"]
    scores = np.dot(X_new_scaled, model_params["W"]) + model_params["b"]
    y_pred_svm = np.where(scores >= 0, 1, -1)
    y_pred_original = np.where(y_pred_svm == 1, 1, 3)
    return y_pred_original